# VLM-Anomaly — Full MVTec Sweep · Qwen3-VL-32B via OpenRouter

**Model:** `qwen/qwen3-vl-32b-instruct`  
**API:** OpenRouter (OpenAI-compatible)  
**Cost estimate:** ~$0.20 for all 1,245 images  
**Runs locally** against your `.env` `OPEN_ROUTER` key and `data/mvtec` dataset.

In [1]:
# ── Cell 1: Setup paths & sys.path ─────────────────────────────────────────
import sys
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT   = Path().resolve().parent   # notebooks/ → repo root
SRC_DIR     = REPO_ROOT / 'src'
PROMPTS_DIR = REPO_ROOT / 'prompts'
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert SRC_DIR.exists(),     f'src/ not found at {SRC_DIR}'
assert PROMPTS_DIR.exists(), f'prompts/ not found at {PROMPTS_DIR}'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(REPO_ROOT / '.env')

import vlm_anomaly
print(f'vlm_anomaly {vlm_anomaly.__version__} ready')
print(f'SRC      : {SRC_DIR}')
print(f'Prompts  : {PROMPTS_DIR}')
print(f'Results  : {RESULTS_DIR}')

vlm_anomaly 0.1.0 ready
SRC      : /Users/sabareeswarans/Projects_26/VLM-Anomaly/src
Prompts  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/prompts
Results  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/results


In [2]:
# ── Cell 2: Verify OpenRouter API key ───────────────────────────────────────
import os

api_key = os.environ.get('OPEN_ROUTER', '')
assert api_key, 'OPEN_ROUTER not set — add it to your .env file'
print(f'OPEN_ROUTER: {api_key[:12]}...{api_key[-4:]}')

OPEN_ROUTER: sk-or-v1-f67...2674


In [3]:
# ── Cell 3: Find MVTec dataset ──────────────────────────────────────────────
MVTEC_ROOT = None
for candidate in [
    REPO_ROOT / 'data' / 'mvtec',
    REPO_ROOT / 'data' / 'mvtec-ad',
    Path('/tmp/mvtec'),
]:
    if candidate.exists() and any(candidate.iterdir()):
        MVTEC_ROOT = candidate
        break

assert MVTEC_ROOT, f'MVTec not found. Expected at {REPO_ROOT}/data/mvtec'
categories = sorted([d.name for d in MVTEC_ROOT.iterdir() if d.is_dir()])
print(f'MVTec root : {MVTEC_ROOT}')
print(f'Categories : {len(categories)} → {categories}')

MVTec root : /Users/sabareeswarans/Projects_26/VLM-Anomaly/data/mvtec
Categories : 15 → ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


In [4]:
# ── Cell 4: Configure ───────────────────────────────────────────────────────
MODEL      = 'qwen/qwen3-vl-32b-instruct'
PROMPT_KEY = 'manufacturing.detailed'
LIMIT      = None    # None = all images; set e.g. 5 for a quick smoke test
BUDGET_USD = 10.0    # hard cap — well above the ~$0.20 expected cost

print(f'Model  : {MODEL}')
print(f'Prompt : {PROMPT_KEY}')
print(f'Limit  : {LIMIT or "all images"}')
print(f'Budget : ${BUDGET_USD}')
print(f'Total  : ~{len(categories) * 83} images')

Model  : qwen/qwen3-vl-32b-instruct
Prompt : manufacturing.detailed
Limit  : all images
Budget : $10.0
Total  : ~1245 images


In [5]:
# ── Cell 5: Build shared objects ────────────────────────────────────────────
from vlm_anomaly.config import Settings
from vlm_anomaly.datasets.mvtec import MVTec
from vlm_anomaly.backends.openrouter import OpenRouterBackend
from vlm_anomaly.evaluators.prompt_library import PromptLibrary
from vlm_anomaly.logging import configure_logging

configure_logging(json_logs=False, log_level='INFO')

settings = Settings(
    _env_file=str(REPO_ROOT / '.env'),
    data_dir=str(MVTEC_ROOT.parent),
    results_dir=str(RESULTS_DIR),
    default_budget_usd=BUDGET_USD,
)
settings.results_dir = RESULTS_DIR

dataset    = MVTec(root_dir=MVTEC_ROOT)
backend    = OpenRouterBackend(model=MODEL)
prompt_lib = PromptLibrary(prompts_dir=PROMPTS_DIR)

print(f'Dataset  : {MVTEC_ROOT}')
print(f'Backend  : {backend.name} / {MODEL}')
print(f'Prompts  : {len(list(prompt_lib._prompts.keys()))} keys loaded')

Dataset  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/data/mvtec
Backend  : openrouter / qwen/qwen3-vl-32b-instruct
Prompts  : 4 keys loaded


In [ ]:
# ── Cell 6: Run all 15 categories ───────────────────────────────────────────
from tqdm.auto import tqdm
from vlm_anomaly.schemas import ExperimentConfig
from vlm_anomaly.evaluators.vlm_evaluator import VLMEvaluator

all_results = []
total_cost  = 0.0

for category in tqdm(categories, desc='MVTec categories'):
    existing = [
        f for f in RESULTS_DIR.glob(f'*_mvtec_{category}.jsonl')
        if 'openrouter' in f.name and f.stat().st_size > 100
    ]
    if existing:
        print(f'  [skip] {category} — already done ({existing[0].name})')
        continue

    config = ExperimentConfig(
        backend=f'openrouter/{MODEL}',
        dataset='mvtec',
        categories=[category],
        prompt=PROMPT_KEY,
        limit=LIMIT,
        budget_usd=BUDGET_USD,
    )
    evaluator = VLMEvaluator(
        backend=backend,
        dataset=dataset,
        config=config,
        settings=settings,
        prompt_library=prompt_lib,
    )
    results = evaluator.run()
    all_results.extend(results)
    for r in results:
        cat_cost = sum(
            p.cost_usd for p in evaluator._last_predictions
            if hasattr(evaluator, '_last_predictions')
        ) if hasattr(evaluator, '_last_predictions') else 0.0
        total_cost += cat_cost
        print(
            f'  {category:12s}  AUROC={r.auroc:.3f}  F1={r.f1:.3f}  '
            f'n={r.n_images}  cost=${r.total_cost_usd:.4f}'
        )

print(f'\nDone. {len(all_results)} categories. Estimated total cost: ${sum(r.total_cost_usd for r in all_results):.4f}')

MVTec categories:   0%|          | 0/15 [00:00<?, ?it/s]

2026-05-23T21:16:51.680272Z [info     ] evaluator.run.start            [vlm_anomaly.evaluators.vlm_evaluator] backend=openrouter budget_usd=10.0 categories=['bottle'] dataset=mvtec experiment_id=5170ed89 limit=None prompt=manufacturing.detailed
2026-05-23T21:16:52.925446Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:16:54.750729Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:16:57.105572Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:16:59.545159Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:17:01.657832Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:17:03.731097Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/complet

  bottle        AUROC=0.209  F1=0.926  n=83  cost=$0.0110


2026-05-23T21:19:52.549305Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:19:54.779290Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:19:57.029782Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:19:58.727079Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:20:01.026208Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:20:03.736939Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:20:05.371588Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:20:07.646192Z [info     ] HTTP Request: POST https://openrouter.ai/ap

  cable         AUROC=0.349  F1=0.618  n=150  cost=$0.0230


...

2026-05-23T21:51:30.724801Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:51:32.714059Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:51:34.824018Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:51:36.804350Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:51:38.814265Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:51:42.166361Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:51:44.350560Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:51:47.417601Z [info     ] HTTP Request: POST https://openrouter.ai/ap

  pill          AUROC=0.482  F1=0.904  n=167  cost=$0.0214


2026-05-23T21:57:51.254999Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:57:53.389141Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:57:55.119042Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:57:57.296651Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:58:02.413536Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:58:04.512181Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:58:06.076055Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T21:58:08.250152Z [info     ] HTTP Request: POST https://openrouter.ai/ap

  screw         AUROC=0.138  F1=0.696  n=160  cost=$0.0245


2026-05-23T22:03:21.259346Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T22:03:23.446761Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T22:03:25.494495Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T22:03:27.436642Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T22:03:29.897368Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T22:03:31.536196Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T22:03:33.526191Z [info     ] HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK" [httpx]
2026-05-23T22:03:35.631530Z [info     ] HTTP Request: POST https://openrouter.ai/ap

In [ ]:
# ── Cell 7: Leaderboard ─────────────────────────────────────────────────────
from vlm_anomaly.analysis.aggregator import leaderboard, cost_accuracy_table

lb = leaderboard(RESULTS_DIR)
if lb.empty:
    print('No results yet.')
else:
    summary = cost_accuracy_table(RESULTS_DIR)
    print('=== Summary (mean across categories) ===')
    print(summary[['model_id', 'mean_auroc', 'mean_latency_ms']].to_string(index=False))
    print()
    print('=== Per-category breakdown (all models) ===')
    display(
        lb[["model_id","category","n_images","auroc","f1","mean_latency_ms"]]
        .sort_values(["model_id","auroc"], ascending=[True,False])
        .reset_index(drop=True)
    )

In [ ]:
# ── Cell 8: Generate report ──────────────────────────────────────────────────
from vlm_anomaly.analysis.report_generator import generate

report = generate(RESULTS_DIR, str(REPO_ROOT / 'REPORT.md'))
print(f'Report written to {report}')

In [ ]:
# ── Cell 9: Show result files + commit hint ──────────────────────────────────
result_files = sorted(RESULTS_DIR.glob('*openrouter*mvtec*.jsonl'))
print(f'Result files ({len(result_files)}):')
for f in result_files:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name}  ({size_kb:.1f} KB)')

print()
print('To commit results:')
print('  git add results/*.jsonl')
print('  git commit -m "results(qwen3-vl): full MVTec sweep via OpenRouter"')